# MorphLens FLUX.1 GGUF Q4 Free Cloud GPU Server

Run this notebook on Google Colab with a **free Tesla T4 GPU** to self-host **FLUX.1 [schnell] GGUF Q4** (~6.5 GB VRAM) with zero OOM errors and connect to your MorphLens app via Cloudflare Tunnel.

### Step 1: Install Dependencies
Click the Play button on this cell. It will install PyTorch diffusers with GGUF support, FastAPI, and Cloudflare tunnel.

In [ ]:
!pip install -q diffusers transformers accelerate sentencepiece protobuf gguf pycloudflared fastapi uvicorn pillow pydantic
!pip install -q git+https://github.com/huggingface/diffusers.git

### Step 2: Launch FLUX.1 GGUF Q4 Server & Cloudflare Tunnel
Click the Play button on this cell. It loads the 6.5 GB GGUF Q4 model into the T4 GPU, launches the FastAPI server with image-to-image support, and generates your public tunnel URL.

In [ ]:
import io
import os
import base64
import random
import threading
import time
from typing import Optional
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
from diffusers import FluxPipeline, FluxImg2ImgPipeline, FluxTransformer2DModel, GGUFQuantizationConfig
import uvicorn
from pycloudflared import try_cloudflare

app = FastAPI(title="MorphLens FLUX.1 GGUF Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("Loading FLUX.1 Schnell GGUF Q4 transformer (~6.5 GB)... (~2-3 mins)")
dtype = torch.float16
transformer = FluxTransformer2DModel.from_single_file(
    "https://huggingface.co/city96/FLUX.1-schnell-gguf/blob/main/flux1-schnell-Q4_K_S.gguf",
    quantization_config=GGUFQuantizationConfig(compute_dtype=dtype),
    torch_dtype=dtype,
)

print("Assembling pipeline components from unsloth/FLUX.1-schnell...")
pipe = FluxPipeline.from_pretrained(
    "unsloth/FLUX.1-schnell",
    transformer=transformer,
    torch_dtype=dtype,
)
pipe.enable_model_cpu_offload()

# Enable zero-copy image-to-image for subject / portrait editing
img2img_pipe = FluxImg2ImgPipeline(**pipe.components)
print("FLUX.1 GGUF Q4 Server Ready!\n")

class GenerateRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = ""
    image: Optional[str] = None
    likeness: Optional[float] = 0.85
    width: Optional[int] = 1024
    height: Optional[int] = 1024
    steps: Optional[int] = 4
    seed: Optional[int] = None

@app.get("/health")
@app.get("/")
def health():
    return {"status": "ok", "model": "FLUX.1-schnell-GGUF-Q4", "ready": True}

def decode_base64_image(data_uri: str) -> Image.Image:
    if "," in data_uri:
        data_uri = data_uri.split(",", 1)[1]
    image_bytes = base64.b64decode(data_uri)
    return Image.open(io.BytesIO(image_bytes)).convert("RGB")

@app.post("/generate")
def generate(req: GenerateRequest):
    try:
        active_seed = req.seed if (req.seed is not None and req.seed >= 0) else random.randint(0, 2**32 - 1)
        generator = torch.Generator("cpu").manual_seed(active_seed)
        
        w = max(256, min(1536, ((req.width or 1024) // 16) * 16))
        h = max(256, min(1536, ((req.height or 1024) // 16) * 16))
        steps = max(1, min(8, req.steps or 4))
        
        if req.image:
            # Subject attached: Image-to-image with likeness preservation
            init_img = decode_base64_image(req.image).resize((w, h), Image.Resampling.LANCZOS)
            likeness_val = max(0.1, min(1.0, req.likeness or 0.85))
            strength = max(0.40, min(0.85, 1.15 - (likeness_val * 0.6)))
            
            output = img2img_pipe(
                prompt=req.prompt,
                image=init_img,
                strength=strength,
                num_inference_steps=steps,
                generator=generator,
                guidance_scale=0.0
            )
        else:
            # Pure text-to-image
            output = pipe(
                prompt=req.prompt,
                width=w,
                height=h,
                num_inference_steps=steps,
                generator=generator,
                guidance_scale=0.0
            )
        
        buffered = io.BytesIO()
        output.images[0].save(buffered, format="JPEG", quality=95)
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        
        return {
            "status": "success",
            "image": f"data:image/jpeg;base64,{img_str}",
            "model": "FLUX.1-schnell-GGUF-Q4",
            "seed": active_seed
        }
    except Exception as e:
        print(f"Generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

port = 8000
tunnel_url = try_cloudflare(port=port)
print("\n" + "="*60)
print("COPY THIS TUNNEL URL INTO MORPHLENS:")
print(tunnel_url.tunnel)
print("="*60 + "\n")

config = uvicorn.Config(app=app, host="0.0.0.0", port=port, log_level="info")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
print("FLUX.1 GGUF Server is live and actively listening for MorphLens!")
while True:
    time.sleep(1)